# SAMPLE — canonical longitudinal workflow (model-agnostic template)

**This notebook is a template, not an experiment, and is not meant to run as-is.**
Every model-specific operation is a placeholder `raise NotImplementedError` hook.
It documents the canonical end-to-end longitudinal pipeline — seed → data → CV →
threshold → save → test → ROC → early-detection → trajectories — shared by all
model notebooks (GELSTM, GEC-MLP, …).

The **SHARED** cells contain no model-specific code: they call the six hooks in the
*Model adapter contract* cell plus reusable helpers under `CLASSIFIER/common/`
(`crossval`, `thresholds`, `plots`, `early_detection`, `run_artifacts`). A concrete
model notebook only fills in the hooks.

Naming: the `BASELINE_/LONGITUDINAL_/STATIC_/SANITY_/COMPARISON_` prefix taxonomy
governs *experiment* notebooks; this template lives in `notebooks/SAMPLE/` with a
`SAMPLE_` prefix to mark its non-experiment status. Cross-cell notebook globals are
UPPERCASE (they act as the notebook's parameters); the seeding handles `rng`/`torch_gen`
and loop-local temporaries stay lowercase.

In [ ]:
# === Papermill parameters (injected by run_experiment.py) ===
# Safe interactive defaults: None keeps the original Jupyter behaviour
# (interactive checkpoint/threshold prompts, JSON-config loading).
EXPERIMENT_ID = None
MODE = None
MODEL = None
DATASET = None
SEED = 42
GAAE_CHECKPOINT_PATH = None   # None -> interactive checkpoint picker
THRESHOLD_MODE = None         # None -> interactive prompt; else youden | best-f1 | fixed
FIXED_THRESHOLD = None        # required when THRESHOLD_MODE is fixed
WANDB_ENABLED = True          # W&B logging is on by default
OUTPUT_DIR = None             # defaults to a sample checkpoints dir when run standalone
RESOLVED_CONFIG = None        # merged hyperparameter dict; overrides on-disk JSON when set
RUN_DIR = None                # set by the runner: where run_summary.json / artifacts go
RUN_NAME = None               # set by the runner: the W&B run name

## Pipeline overview

`set_seed` → load splits → `prepare_data` (HOOK) → `build_model` (HOOK) →
`run_kfold_cv` (shared, drives the `train_fold` HOOK) → `select_oof_threshold` (shared) →
`save_run` (shared) → test-set evaluation (`eval_split` HOOK) → `plot_oof_test_roc` →
`early_detection_table` (`truncate_to_n_visits` HOOK) →
`trajectory_frame` + `plot_conversion_trajectories` (`per_visit_probs` HOOK).

In [ ]:
import sys
from pathlib import Path
repo_root = Path('/mnt/e/fyassine/ad-early-detection')
model_root = Path('/mnt/e/fyassine/ad-early-detection/CLASSIFIER')
if str(model_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
    sys.path.insert(0, str(model_root))

In [ ]:
# reproducibility seeding — must run before datasets, samplers, or models.
from CLASSIFIER.common.seeding import (
    set_seed, make_rng, make_torch_generator, seed_worker,
)
set_seed(SEED)
rng = make_rng(SEED)
torch_gen = make_torch_generator(SEED)

In [ ]:
import json, os, copy, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from datetime import datetime
from sklearn.metrics import classification_report

# Shared, model-agnostic utilities (the lifted notebook logic — reuse, do not inline).
from common.checkpoints import select_gaae_checkpoint
from common.sanity import run_full_audit
from common import tracking
from common.provenance import region_from_data_root
from common.crossval import Bundle, run_kfold_cv, summarize_cv
from common.thresholds import select_oof_threshold
from common.plots import plot_oof_test_roc, plot_conversion_trajectories
from common.early_detection import early_detection_table, trajectory_frame
from common.run_artifacts import save_run, record_test_metrics

# ── MODEL-SPECIFIC IMPORTS (fill in per model notebook) ───────────────────
# e.g. GELSTM:
#   from model.GELSTM.models  import GELSTMClassifier
#   from model.GELSTM.dataset import LongitudinalSubjectDataset
#   from model.GELSTM.train   import train_model, evaluate, make_batches
#   from model.GELSTM.utils   import encode_batch_sequences, compute_class_weights
# e.g. GEC (flat MLP):
#   from model.GAAE.models   import GraphAttentionAutoencoderConditioned
#   from model.GELSTM.dataset import LongitudinalSubjectDataset

warnings.filterwarnings('ignore')
print('Imports OK')

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## Model adapter contract

A concrete model notebook implements the six hooks below (replacing each
`raise NotImplementedError`). The SHARED cells call only these hooks and the
`common.crossval.Bundle` container, so they remain identical across models:

| Hook | Responsibility |
| --- | --- |
| `build_model()` | instantiate model, load + freeze GAAE encoder |
| `prepare_data(df)` | encode a split into a `Bundle` |
| `train_fold(tr, va, cfg)` | train + select on one CV fold |
| `eval_split(state, bundle, thr)` | score a split at a fixed threshold |
| `truncate_to_n_visits(bundle, n)` | first-N-visits view (early detection) |
| `per_visit_probs(state, item)` | per-visit prefix probabilities (trajectories) |

In [ ]:
# === MODEL-SPECIFIC HOOKS ====================================================
# This template is model-agnostic: every model-specific operation is funneled
# through the functions below. A concrete model notebook replaces each
# `raise NotImplementedError` body; the SHARED cells call ONLY these hooks (and
# the imported `Bundle`), so they stay identical across models.
#
# `Bundle` is `common.crossval.Bundle` (imported above): a container exposing
# .labels / .groups / .items / .subset(idx). `prepare_data` must return one.

def build_model() -> 'nn.Module':
    """Instantiate the model (load + freeze the GAAE encoder), print param counts."""
    raise NotImplementedError('per-model hook: build and return the model')


def prepare_data(df: 'pd.DataFrame') -> Bundle:
    """Encode a split DataFrame into a `common.crossval.Bundle`.

    GELSTM: wrap LongitudinalSubjectDataset items directly.
    GEC:    encode each visit through the frozen GAAE, flatten to fixed vectors,
            and carry the per-subject payload inside each Bundle item.
    Each item must expose at least 'subject_id', 'label', 'n_scans'.
    """
    raise NotImplementedError('per-model hook: encode df into a Bundle')


def train_fold(bundle_tr: Bundle, bundle_va: Bundle, cfg: dict, *, rng, device) -> dict:
    """Train one CV fold (called by common.crossval.run_kfold_cv). Returns:
        {
          'state_dict':     best model state (val-AUC selected),
          'val_metrics':    {'auc','sensitivity','specificity','f1'},
          'best_threshold': float (val-derived),
          'oof_probs':      np.ndarray, 'oof_targets': np.ndarray, 'oof_sids': list[str],
        }
    """
    raise NotImplementedError('per-model hook: train + select on the val fold')


def eval_split(state_dict, bundle: Bundle, threshold: float, *, device) -> dict:
    """Evaluate a trained model on a bundle at a FIXED threshold (never re-optimised
    on the data being scored — see .claude/rules/evaluation.md). Returns:
        {'auc','sensitivity','specificity','f1','probs','targets','subject_ids','n_scans'}
    """
    raise NotImplementedError('per-model hook: score bundle at the given threshold')


def truncate_to_n_visits(bundle: Bundle, n: int) -> Bundle:
    """Return a Bundle with each subject restricted to their first `n` visits
    (subjects with fewer than `n` visits are dropped). Used by early_detection_table.
    """
    raise NotImplementedError('per-model hook: first-N-visits truncation')


def per_visit_probs(state_dict, item: dict, *, device) -> list:
    """Return [(visit_month, P(converter)), ...] for prefix sequences of length 1..T
    of a single subject `item`. Used by trajectory_frame.
    """
    raise NotImplementedError('per-model hook: per-visit prefix probabilities')

## Configuration

In [ ]:
from DATA.src.splitting.load_splits import splits_dir, split_csv_paths

# ── Paths ────────────────────────────────────────────────────────────────
WB_DATA_ROOT = '/mnt/e/fyassine/ad-early-detection/DATA/DELCODE/__fc_wholebrain_sch200_flat__/matrices'
METADATA_DIR = '/mnt/e/fyassine/ad-early-detection/DATA/DELCODE/__fc_wholebrain_sch200_flat__/metadata'
COHORTS_CSV  = os.path.join(METADATA_DIR, 'cohorts.csv')
SPLITS_DIR   = str(splits_dir('downstream'))
TRAIN_CSV    = os.path.join(SPLITS_DIR, 'train.csv')
VAL_CSV      = os.path.join(SPLITS_DIR, 'val.csv')
TEST_CSV     = os.path.join(SPLITS_DIR, 'test.csv')

# Brain region / atlas parsed from the DATA directory name (surfaced in run name + config).
DATA_INFO = region_from_data_root(WB_DATA_ROOT)
REGION    = DATA_INFO['region']
print(f"Input data: region={DATA_INFO['region']}  atlas={DATA_INFO['atlas']}  ({DATA_INFO['dataset_dir']})")

CHECKPOINT_SEARCH_DIRS = [
    str(model_root / 'notebooks' / 'checkpoints' / 'checkpoints_gaae_whole_brain'),
]
if OUTPUT_DIR is None:
    OUTPUT_DIR = str(model_root / 'notebooks' / 'checkpoints' / 'checkpoints_sample_template')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# GAAE encoder config (must match checkpoint).
CONFIG_PATH = model_root / 'configs' / 'gaae_delcode_whole_brain.json'

# ── MODEL-SPECIFIC: training hyper-params ─────────────────────────────────
# Point TRAIN_CONFIG_PATH at this model's configs/*.json. The load + RESOLVED_CONFIG
# merge below is shared boilerplate; only the path and the hyperparam unpacking change.
TRAIN_CONFIG_PATH = model_root / 'configs' / '<model>_delcode_whole_brain.json'   # EDIT per model
_train_defaults = {'epochs': 50, 'batch_size': 16, 'learning_rate': 1e-3,
                   'grad_clip': 1.0, 'early_stopping_patience': 15, 'n_folds': 5}
if TRAIN_CONFIG_PATH.exists():
    with open(TRAIN_CONFIG_PATH) as _f:
        TRAIN_CONFIG = json.load(_f)
    print(f'Loaded training config from {TRAIN_CONFIG_PATH}')
else:
    TRAIN_CONFIG = _train_defaults
    print(f'Training config not found at {TRAIN_CONFIG_PATH} — using inline defaults.')

# Runner override: merge injected RESOLVED_CONFIG (YAML hyperparams) over JSON config.
if RESOLVED_CONFIG:
    TRAIN_CONFIG = {**TRAIN_CONFIG, **RESOLVED_CONFIG}
    print('Applied RESOLVED_CONFIG overrides from runner.')

N_FOLDS = TRAIN_CONFIG.get('n_folds', 5)
print('Config set.')

In [ ]:
# split-hygiene audit — hard-fails if any subject crosses splits.
_ = run_full_audit(split_csv_paths('downstream'))

## Select GAAE Checkpoint

In [ ]:
# Shared helper (already in common/checkpoints.py): lists candidates and prompts for
# an index interactively, or resolves GAAE_CHECKPOINT_PATH headlessly under the runner.
if GAAE_CHECKPOINT_PATH is None and RUN_DIR is not None:
    raise ValueError(
        "GAAE_CHECKPOINT_PATH is required under the experiment runner. "
        "Set 'checkpoint_path:' on this entry in experiments.yaml."
    )
GAAE_RUN_NAME, GAAE_CKPT_PATH, GAAE_RUN_DIR = select_gaae_checkpoint(
    CHECKPOINT_SEARCH_DIRS, checkpoint_path=GAAE_CHECKPOINT_PATH,
)
print(f'Selected GAAE: {GAAE_RUN_NAME}')

In [ ]:
if CONFIG_PATH.exists():
    with open(CONFIG_PATH) as f: hp = json.load(f)
    print('GAAE config:', hp)
else:
    hp = dict(latent_dim=64, hidden_dim=128, num_heads=2, cond_dim=2, dropout=0.3,
              adjacency_k=8, file_variant='z_transformed')
    print('GAAE config not found — using defaults.')

IN_FEATURES   = 200   # Schaefer-200 ROIs
GAAE_HIDDEN   = hp.get('hidden_dim', 128)
GAAE_LATENT   = hp.get('latent_dim', 64)
GAAE_HEADS    = hp.get('num_heads', 2)
GAAE_COND_DIM = hp.get('cond_dim', 2)
GAAE_DROPOUT  = hp.get('dropout', 0.3)
ADJ_K         = hp.get('adjacency_k', 8)
FILE_VARIANT  = hp.get('file_variant', 'z_transformed')
print(f'GAAE hidden={GAAE_HIDDEN}  latent={GAAE_LATENT}  heads={GAAE_HEADS}')

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

# CV pool = train + val; test held out.
cv_pool_df = pd.concat([train_df, val_df], ignore_index=True)

print('CV pool:', cv_pool_df['diagnosis'].value_counts().to_dict())
print('Test:   ', test_df['diagnosis'].value_counts().to_dict())

In [ ]:
# HOOK calls: encode each split into a model-specific Bundle, then smoke-test the model.
CV_BUNDLE   = prepare_data(cv_pool_df)
TEST_BUNDLE = prepare_data(test_df)
print(f'CV subjects: {len(CV_BUNDLE)}  Test subjects: {len(TEST_BUNDLE)}')

_ = build_model()   # smoke-test arch construction

## 5-Fold Stratified Subject-Level Cross-Validation

In [ ]:
# Open the W&B run, then run the shared CV loop (common.crossval.run_kfold_cv) driven
# by the train_fold hook. Per-fold metrics flow to W&B via the log_fn callback, so the
# loop itself stays W&B-free.
_wb_exp = {'id': EXPERIMENT_ID or 'sample-template', 'mode': MODE or 'longitudinal',
           'model': MODEL or 'SAMPLE', 'dataset': DATASET or REGION,
           'seed': SEED, 'wandb': WANDB_ENABLED}
WANDB_RUN = tracking.init_run(_wb_exp, {**(RESOLVED_CONFIG or {}), 'REGION': REGION})

CV = run_kfold_cv(
    CV_BUNDLE, train_fold, TRAIN_CONFIG,
    n_folds=N_FOLDS, rng=rng, device=device,
    log_fn=lambda d: tracking.log_metrics(WANDB_RUN, d),
)

CV_RESULTS             = CV.cv_results
OOF_PROBS, OOF_TARGETS = CV.oof_probs, CV.oof_targets
OOF_SIDS               = CV.oof_sids
BEST_MODEL_STATE       = CV.best_model_state
BEST_FOLD              = CV.best_fold
BEST_VAL_AUC           = CV.best_val_auc
BEST_THRESHOLD_OVERALL = CV.best_threshold
BEST_F1_THR            = CV.best_f1_threshold

## Cross-Validation Summary

In [ ]:
summarize_cv(CV_RESULTS)

In [ ]:
# Resolve the active threshold from the pooled out-of-fold predictions (Best-F1 default).
# Validation/OOF only — never the test set (see .claude/rules/evaluation.md).
ACTIVE_THRESHOLD, THRESHOLD_METHOD = select_oof_threshold(
    OOF_TARGETS, OOF_PROBS,
    threshold_mode=THRESHOLD_MODE, fixed_threshold=FIXED_THRESHOLD,
    runner_active=RUN_DIR is not None,
)

## Save Best Model

In [ ]:
# ── MODEL-SPECIFIC: architecture description + source files for this run ────
MODEL_CONFIG = {
    'model_type': '<ModelClass>',           # EDIT per model
    'in_features': IN_FEATURES,
    'gaae_hidden': GAAE_HIDDEN, 'gaae_latent': GAAE_LATENT,
    'gaae_heads': GAAE_HEADS, 'gaae_cond_dim': GAAE_COND_DIM, 'gaae_dropout': GAAE_DROPOUT,
    # ... add this model's own arch fields (lstm_hidden / mlp_hidden_layers / ...)
}
SOURCE_FILES = [
    model_root / 'model' / 'GAAE' / 'models.py',
    # model_root / 'model' / '<FAMILY>' / 'models.py',
    # model_root / 'model' / '<FAMILY>' / 'train.py',
]
DATASET_INFO = {**DATA_INFO, 'train_csv': TRAIN_CSV, 'val_csv': VAL_CSV,
                'test_csv': TEST_CSV, 'n_folds': N_FOLDS}

# Shared run-saver: make_run_dir → state dict + full-state checkpoint → source snapshot
# → run_summary.json. Returns the resolved (RUN_NAME, RUN_DIR).
RUN_NAME, RUN_DIR = save_run(
    output_dir=OUTPUT_DIR, run_dir=RUN_DIR, run_name=RUN_NAME,
    model_state=BEST_MODEL_STATE, model_config=MODEL_CONFIG, training_config=TRAIN_CONFIG,
    data_info=DATA_INFO, dataset_info=DATASET_INFO, rng=rng,
    best_val_auc=BEST_VAL_AUC, active_threshold=ACTIVE_THRESHOLD, threshold_method=THRESHOLD_METHOD,
    best_fold=BEST_FOLD, cv_results=CV_RESULTS,
    gaae_checkpoint=GAAE_CKPT_PATH, gaae_run_name=GAAE_RUN_NAME,
    source_files=SOURCE_FILES, n_folds=N_FOLDS, model_tag='sample',
)
print(f'Saved to {RUN_DIR}')

try:
    tracking.log_metrics(WANDB_RUN, {'cv_best_val_auc': float(BEST_VAL_AUC),
                                     'active_threshold': float(ACTIVE_THRESHOLD)})
except NameError:
    pass

## Test-Set Evaluation

In [ ]:
# HOOK: score the held-out test set at the val-derived threshold (no test leakage).
TEST_METRICS = eval_split(BEST_MODEL_STATE, TEST_BUNDLE, ACTIVE_THRESHOLD, device=device)

print('Test-Set Results')
print('=' * 40)
print(f"AUC:         {TEST_METRICS['auc']:.4f}")
print(f"Sensitivity: {TEST_METRICS['sensitivity']:.4f}")
print(f"Specificity: {TEST_METRICS['specificity']:.4f}")
print(f"F1:          {TEST_METRICS['f1']:.4f}")
print(f"Threshold:   {ACTIVE_THRESHOLD:.4f}  ({THRESHOLD_METHOD})")
print()
print(classification_report(TEST_METRICS['targets'],
                            (np.asarray(TEST_METRICS['probs']) >= ACTIVE_THRESHOLD).astype(int),
                            target_names=['stable_mci', 'converter']))

# Shared: patch run_summary.json with the standard test-metric schema.
record_test_metrics(RUN_DIR, TEST_METRICS, threshold=ACTIVE_THRESHOLD, threshold_method=THRESHOLD_METHOD)
print(f"Test metrics saved to {RUN_DIR / 'run_summary.json'}")

try:
    tracking.log_metrics(WANDB_RUN, {'test_auc': float(TEST_METRICS['auc']), 'test_f1': float(TEST_METRICS['f1']),
                                     'test_sensitivity': float(TEST_METRICS['sensitivity']),
                                     'test_specificity': float(TEST_METRICS['specificity'])})
    tracking.finish_run(WANDB_RUN)
except NameError:
    pass

## ROC Curves

In [ ]:
fig = plot_oof_test_roc(
    OOF_TARGETS, OOF_PROBS, TEST_METRICS['targets'], TEST_METRICS['probs'],
    title=f'SAMPLE template  |  GAAE: {GAAE_RUN_NAME}',
)
plt.show()

## Early-Detection Curve: AUC vs. Number of Visits Used

Shared `early_detection_table`: for each N it restricts every test subject with `T >= N`
visits to their first N (via the `truncate_to_n_visits` hook) and re-scores with
`eval_split`. Rows with fewer than 4 subjects or a single class are skipped. `N=1` =
single-scan / baseline-only classification.

In [ ]:
ED_ROWS = early_detection_table(
    TEST_BUNDLE, eval_split, truncate_to_n_visits,
    BEST_MODEL_STATE, ACTIVE_THRESHOLD, device=device,
)

## Conversion Probability Trajectories

Shared `trajectory_frame` builds per-visit `P(converter)` for each test subject with
>= 2 visits (via the `per_visit_probs` hook); `plot_conversion_trajectories` renders the
converter / stable panels.

In [ ]:
TRAJ_DF = trajectory_frame(TEST_BUNDLE, per_visit_probs, BEST_MODEL_STATE, device=device)
print(f"Trajectory data: {TRAJ_DF['pid'].nunique()} subjects")
fig = plot_conversion_trajectories(
    TRAJ_DF, ACTIVE_THRESHOLD,
    title='Per-visit conversion probability trajectories  |  SAMPLE template',
)
plt.show()